## Install chromadb



In [ ]:
# We need 3 libraries:
# sentence-transformers → to convert text into numbers (embeddings)
# chromadb             → to store and search those numbers (vector index)
# pandas               → to handle our dataset easily

!pip install chromadb

## Import Libaries

In [ ]:
# Bringing the tools into our code

from sentence_transformers import SentenceTransformer
import chromadb
import pandas as pd

## Create Your Dataset

In [ ]:
# This is our sample dataset
# Think of it as a small company's HR/IT policy knowledge base
# You can replace these with your own later

documents = [
    "Employees can request a laptop replacement after 3 years of use.",
    "To report an incident, fill the form on the HR portal within 24 hours.",
    "Reimbursement requests must be submitted within 30 days of the expense.",
    "Work from home is allowed up to 3 days a week with manager approval.",
    "Medical leave can be taken for up to 15 days per year with a doctor certificate.",
    "New employees get onboarding training in their first week.",
    "IT support can be reached at support@company.com or extension 1234.",
    "Annual performance reviews happen every December.",
]

# Let's also give each document a unique ID
# ChromaDB needs IDs to store documents
doc_ids = ["doc1", "doc2", "doc3", "doc4", "doc5", "doc6", "doc7", "doc8"] # These are the list of the id that we create and can use by chromadb

print("Total documents in dataset:", len(documents))
print("First document:", documents[0])

Total documents in dataset: 8
First document: Employees can request a laptop replacement after 3 years of use.


## Load The Embedding Model

In [ ]:
# We are using a pre-trained model called 'all-MiniLM-L6-v2'
# This model knows how to read text and convert it into numbers
# We are NOT training it — we are just USING it

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded successfully!")
print("This model converts any text into a vector of 384 numbers")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
This model converts any text into a vector of 384 numbers


## Generate Embeddings

In [ ]:
# Now we pass each document through the model
# The model reads each sentence and gives back a list of numbers
# Those numbers CAPTURE THE MEANING of the sentence

embeddings = model.encode(documents)

# Let's see what one embedding looks like
print("Total embeddings created:", len(embeddings))
print("Shape of one embedding:", embeddings[0].shape)
print("First few numbers of first embedding:", embeddings[0][:5])

Total embeddings created: 8
Shape of one embedding: (384,)
First few numbers of first embedding: [-0.05683772  0.02988028  0.0717371  -0.06836649  0.0031189 ]


## Create A Vector Index (chromadb)

In [ ]:
# ChromaDB is our vector database
# It stores our documents + their embeddings together
# So later when we search, it can find the most similar ones

# Step 1: Start ChromaDB in memory (no file needed)
client = chromadb.Client()

# Step 2: Create a "collection" — think of it like a table in SQL
collection = client.create_collection(name = "my_knowledge_base")

print("ChromaDB started!")
print("Empty collection created: my_knowledge_base")

ChromaDB started!
Empty collection created: my_knowledge_base


## Store Everything In The Index

In [ ]:
# Now we put our documents AND their embeddings into ChromaDB
# ChromaDB stores:
#   - The original text (documents)
#   - The numeric vectors (embeddings)
#   - The IDs (doc_ids)

# We need to convert embeddings to a plain list format for ChromaDB
embeddings_as_list = embeddings.tolist()

# Now add everything to the collection
collection.add(
    documents = documents,
    embeddings = embeddings_as_list,
    ids = doc_ids
)

print("All documents stored in the vector index!")
print("Total documents in index:", collection.count())

All documents stored in the vector index!
Total documents in index: 8


## Search With A Clean Query (No Typo)

In [ ]:
# Let's test a normal search first
# We type a question and ChromaDB finds the most similar documents

user_query = "how do I get a new laptop"

# Step 1: Convert the query into an embedding (same model)
query_embedding = model.encode([user_query]).tolist()

# Step 2: Ask ChromaDB to find top 3 most similar documents
results = collection.query(
    query_embeddings = query_embedding,
    n_results = 3
)

print("Query:", user_query)
print("---")
print("Top 3 Results:")
for i, doc in enumerate(results['documents'][0]):
    print(f"Result {i+1}: {doc}")

Query: how do I get a new laptop
---
Top 3 Results:
Result 1: Employees can request a laptop replacement after 3 years of use.
Result 2: IT support can be reached at support@company.com or extension 1234.
Result 3: New employees get onboarding training in their first week.


## Search With A Typo (This Is Where SQL Fails!)

In [ ]:
# Now let's try the same query BUT with a typo
# SQL would return ZERO results for this
# Our system should still find the right answer

user_query_with_typo = "how do I get a nwe laptoop"  # typos: nwe, laptoop

# Step 1: Convert the typo query into an embedding
query_embedding_typo = model.encode([user_query_with_typo]).tolist()

# Step 2: Search the index
results_typo = collection.query(
    query_embeddings = query_embedding_typo,
    n_results = 3
)

print("Query WITH typo:", user_query_with_typo)
print("---")
print("Top 3 Results (even with typo!):")
for i, doc in enumerate(results_typo['documents'][0]):
    print(f"Result {i+1}: {doc}")

Query WITH typo: how do I get a nwe laptoop
---
Top 3 Results (even with typo!):
Result 1: IT support can be reached at support@company.com or extension 1234.
Result 2: Employees can request a laptop replacement after 3 years of use.
Result 3: New employees get onboarding training in their first week.


# WHAT IS A SIMILARITY SCORE?

When we convert text into embeddings (numbers/vectors),
each piece of text becomes a point in a high-dimensional space.

Similarity Score tells us:
> "How CLOSE are two points in that space?"

| Score | Meaning |
|-------|---------|
| 1.0   | Identical meaning |
| 0.8   | Very similar meaning |
| 0.5   | Somewhat related |
| 0.2   | Very different meaning |
| 0.0   | Completely opposite |

---

# HOW IS IT CALCULATED?

We use something called **Cosine Similarity**
It measures the **ANGLE** between two vectors

- Small angle  →  Similar direction  →  **High score**
- Large angle  →  Different direction →  **Low score**

**Think of it like this:**
- Two people walking in almost the **SAME direction** = similar
- Two people walking in **OPPOSITE directions** = different

---

# WHAT DOES IT EVALUATE?

It evaluates **MEANING**, not exact words.

**Example 1 — High Score:**
- "I want a laptop"
- "I need a computer"
- → Same meaning, different words = **High similarity score**

**Example 2 — Low Score:**
- "I want a laptop"
- "What is the leave policy"
- → Completely different meaning = **Low similarity score**

---

# WHY IS THIS BETTER THAN SQL?

| | SQL | Embeddings |
|---|---|---|
| Checks | Exact characters | Meaning |
| Typos | ❌ Fails | ✅ Works |
| Synonyms | ❌ Fails | ✅ Works |
| Different phrasing | ❌ Fails | ✅ Works |

# WHAT IS enumerate()?

`enumerate()` is a helper that gives you **TWO things at once:**

1. The **COUNT** (position number)
2. The **ITEM** (the actual value)

**Example:**
```python
fruits = ["apple", "banana", "mango"]

for i, fruit in enumerate(fruits):
    print(i, fruit)
```
**Output:**
```
0  apple
1  banana
2  mango
```
> ⚠️ Notice: counting starts from **0** by default
> (computers count from 0, not 1)

---

# HOW TO START COUNTING FROM 1 INSTEAD OF 0?

Just tell `enumerate()` where to start:

```python
for i, fruit in enumerate(fruits, start=1):
    print(i, fruit)
```
**Output:**
```
1  apple
2  banana
3  mango
```

---

# WITHOUT enumerate() vs WITH enumerate()

| | Code | What you get |
|---|---|---|
| **WITHOUT** | `for fruit in fruits:` | Only the item, no number |
| **WITH** | `for i, fruit in enumerate(fruits):` | Number AND item together |

---

# REAL LIFE EXAMPLE (Like our similarity results)

```python
results = ["Laptop Policy", "Leave Policy", "IT Support"]
```

**WITHOUT enumerate:**
```python
for result in results:
    print(result)
```
```
Laptop Policy
Leave Policy
IT Support
```

**WITH enumerate:**
```python
for i, result in enumerate(results, start=1):
    print("Rank", i, ":", result)
```
```
Rank 1 : Laptop Policy
Rank 2 : Leave Policy
Rank 3 : IT Support
```

> That is exactly how we showed **Rank 1, Rank 2, Rank 3**
> in our similarity search results!

---

# ONE LINE SUMMARY

> **`enumerate()`  =  for loop  +  automatic counter**
